# 50 Analysis (CPU runtime)

Figures for the report from a pipeline run's pool, diversity and verification outputs.

**After every code update, reopen this notebook from the GitHub link** (README badge): an already open Colab tab keeps the old cells even though the setup cell pulls the new code.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
RUN = "pool_smoke"
import json, pandas as pd, matplotlib.pyplot as plt
R = f"{DRIVE_ROOT}/effsql/runs/{RUN}"
pool = pd.read_parquet(f"{R}/pool_train.parquet")
summary = json.load(open(f"{R}/summary_train.json"))
pool.head()

## Efficiency score of correct candidates (positive = cheaper than gold)

In [ ]:
s = pool.loc[pool["correct"] & ~pool["is_gold"], "eff_score"].dropna()
plt.hist(s, bins=60); plt.axvline(0, color="k", lw=1)
plt.xlabel("eff_score = log2(gold steps / candidate steps)"); plt.ylabel("candidates"); plt.show()

## Correct rate by source

In [ ]:
src = pool[pool["error_type"] != "parse_fail"].assign(src=lambda d: d["sources"].str.split(";")).explode("src")
src.groupby("src")["correct"].mean().sort_values().plot.barh(); plt.xlabel("share correct"); plt.show()

## Diversity before and after rewriting

In [ ]:
d = pd.DataFrame(summary["diversity_rounds"]).set_index("round")
d["pct_diverse_of_usable"].plot.bar(); plt.ylabel("% of usable questions diverse"); plt.show()
d[["reasons"]]

## Steps vs real time (verification sample)

In [ ]:
rows = [c for line in open(f"{R}/verify_train.jsonl") for c in json.loads(line)["candidates"]]
v = pd.DataFrame(rows)
plt.loglog(v["vm_steps"] + 10, v["median_s"], ".", alpha=0.5)
plt.xlabel("vm_steps + 10"); plt.ylabel("median time (s)"); plt.show()
summary["verify"]